# Extract Packet Features & Create Chronological Train/Val Split

Reads the training PCAPs (benign & DoS), extracts features using the shared `ble_extractor`,  
splits each file by time (first 80% for training, last 20% for validation), and saves separate  
CSV files: `ble_train.csv` and `ble_val.csv`.  
This prevents data leakage and yields an honest validation metric.

In [7]:
import sys, os
sys.path.append('..')
import pandas as pd
import numpy as np

## 1. Extract packets from both training PCAPs

In [8]:
from scapy.all import rdpcap, Packet
import pandas as pd
import numpy as np
import os

def extract_packets_from_pcap(pcap_path, label=None):
    """
    Extract packet features from a BLE pcap file using scapy.
    Returns a list of dicts with keys: timestamp, info, length, delta, type (if label given).
    """
    packets = rdpcap(pcap_path)
    records = []
    prev_time = None

    for pkt in packets:
        try:
            # Use scapy's summary as a text description (similar to Wireshark Info)
            info = pkt.summary()
            length = len(pkt)
            ts = float(pkt.time)
            delta = 0.0 if prev_time is None else ts - prev_time
            prev_time = ts
            rec = {
                'timestamp': ts,
                'info': str(info),
                'length': length,
                'delta': delta
            }
            if label is not None:
                rec['type'] = label
            records.append(rec)
        except Exception as e:
            # skip corrupted packets
            continue
    return records

In [9]:
# Paths (relative to notebooks/)
benign_pcap = '../data/raw/ble_pcaps/Bluetooth_Benign_train.pcap'
dos_pcap    = '../data/raw/ble_pcaps/Bluetooth_DoS_train.pcap'

# No tshark_path needed anymore
benign_recs = extract_packets_from_pcap(benign_pcap, label='normal')
dos_recs    = extract_packets_from_pcap(dos_pcap, label='DoS')

print(f"Benign packets: {len(benign_recs)}")
print(f"DoS packets:     {len(dos_recs)}")

Benign packets: 217493
DoS packets:     998391


## 2. Chronological split (80% train / 20% val) per class

Packets are extracted in chronological order. We'll keep the first 80% of each file for training  
and the last 20% for validation, then combine.

In [10]:
def split_records(records, train_frac=0.8):
    n = len(records)
    split_idx = int(n * train_frac)
    return records[:split_idx], records[split_idx:]

benign_train, benign_val = split_records(benign_recs)
dos_train, dos_val       = split_records(dos_recs)

print("Benign: train =", len(benign_train), ", val =", len(benign_val))
print("DoS:    train =", len(dos_train), ", val =", len(dos_val))

Benign: train = 173994 , val = 43499
DoS:    train = 798712 , val = 199679


## 3. Combine and save as CSV

In [11]:
train_df = pd.DataFrame(benign_train + dos_train)
val_df   = pd.DataFrame(benign_val + dos_val)

# Shuffle training data (optional, but keep chronological order in val)
train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)

train_df.to_csv('../data/raw/ble_train.csv', index=False)
val_df.to_csv('../data/raw/ble_val.csv', index=False)

print("Train set:", train_df['type'].value_counts().to_dict())
print("Val set:  ", val_df['type'].value_counts().to_dict())

Train set: {'DoS': 798712, 'normal': 173994}
Val set:   {'DoS': 199679, 'normal': 43499}
